# NTT-PEEL — Toy (N = 8, q = 17) Guess-and-Prune Attack

A **self-contained, CPU-only, pedagogical downscaling** of the single-trace side-channel
attack from

> **NTT-PEEL: Bit Shift Side-Channel in FALCON's Number Theoretic Transform**
> Jinyi Qiu and Aydin Aysu — *IACR TCHES 2026(2), pp. 709-735* — DOI:10.46586/tches.v2026.i2.709-735

The published artifact attacks degree-512 polynomials over $q = 12289$ and needs a CUDA GPU, ~1.5 TB of
disk, and up to two weeks of runtime. This notebook keeps the attack mechanism faithful but shrinks the
transform to an **8-coefficient polynomial with 4 butterflies per stage** (3 NTT stages) over a **small
modulus $q = 17$**, so the whole attack runs on a laptop CPU in seconds and is visible end-to-end.

**Everything is merged into this one notebook:**

1. **Gaussian sampling** of the secret polynomial (σ rescaled to the small modulus).
2. **In-place modular reduction flag** simulation and generation (the leaky `mq_NTT`).
3. **Forward pruning** — Algorithm 2 of the paper.
4. **Backward refinement** — Algorithm 3 of the paper.
5. **Input-coefficient-space exploration** — search the key space and select the 8-coefficient polynomial that yields the *best* reduction.
6. **Exact key recovery** over the pruned space, plus a brute-force correctness check.

At $q = 17$ the attack is **over-determined** (more leaked flag-bits than secret-bits), so favourable
keys are recovered **completely** from a single trace — the same success regime as the real FALCON-512
attack.


## Why q = 17?

The size-$N$ **negacyclic** NTT (the FALCON structure, with the convolution property we verify below)
needs a primitive $2N$-th root of unity, so for $N = 8$ we need a $16$-th root, i.e.

$$16 \mid (q-1) \quad\Longleftrightarrow\quad q \equiv 1 \pmod{16}.$$

The small primes that *do* qualify are
$17, 97, 113, 193, \dots$. Among them **smaller is better here**, for two reasons:

* **Flag balance.** The attack's information comes from reduction flags that fire roughly half the time.
  As $q$ grows, the twiddle-multiplication flag *saturates* (it fires ~87–98 % of the time at $q\ge 97$),
  so each flag carries almost no information. At $q = 17$ the flags stay balanced
  (tw ≈ 69 %, add ≈ 37 %, sub ≈ 46 %) — informative, and *not* saturated near the modulus.
* **Over-determined recovery.** With 3 stages there are $4\cdot3\cdot3 = 36$ leaked flag-bits. At
  $q = 17$ a key carries only ~25 bits, so the flags over-determine it and the best keys are recovered
  *entirely*. (At $q = 12289$ the same $N=8$ key carries ~38 bits > 36, so it could not be fully pinned.)


## 0 · Imports and global parameters

In [178]:
import random, math, itertools, statistics, time
import numpy as np

try:
    import matplotlib.pyplot as plt
    HAVE_PLT = True
except Exception:
    HAVE_PLT = False

# ---- toy field / transform parameters ----------------------------------------
q      = 17                               # small modulus, q ≡ 1 (mod 16) so a 16th root exists
N      = 8                                # transform size (FALCON-512 uses 512)
logN   = 3                               # number of NTT stages = log2(N)
SIGMA  = 1.17 * math.sqrt(q / (2 * N))   # FALCON's key-gen sigma formula, evaluated at (q,N) -> ~1.206
B      = math.ceil(3.2 * SIGMA)          # attacker guess half-width: candidates {-B,...,B} mod q

assert (q - 1) % 16 == 0, "need q ≡ 1 (mod 16) for a size-8 negacyclic NTT"
print(f"q={q}  N={N}  logN={logN}  sigma={SIGMA:.3f}  guess range=[-{B},{B}] ({2*B+1} values/coeff)")


q=17  N=8  logN=3  sigma=1.206  guess range=[-4,4] (9 values/coeff)


## 1 · The twiddle table (built the FALCON way, for q = 17)

FALCON's NTT is a **negacyclic**, radix-2, decimation-in-frequency Cooley-Tukey transform over
$\mathbb{Z}_q[x]/(x^N+1)$, with twiddle table $\texttt{GMb}[i] = \psi^{\mathrm{bitrev}(i)}$ where $\psi$
is a primitive $2N$-th root of unity. We build the same table for $q = 17$: the primitive $16$-th root is
$\psi = 3$, giving $\texttt{GMb} = [1, 13, 9, 15, 3, 5, 10, 11]$.

In [179]:
def order(a, mod):
    o, x = 1, a % mod
    while x != 1:
        x = (x * a) % mod; o += 1
    return o

def bitrev(i, bits):
    r = 0
    for b in range(bits):
        r = (r << 1) | ((i >> b) & 1)
    return r

# primitive 16th root of unity mod q: g^((q-1)/16) for the first g that gives order exactly 16
psi16 = next(c for c in (pow(g, (q - 1) // 16, q) for g in range(2, q)) if order(c, q) == 16)
GMb   = [pow(psi16, bitrev(i, logN), q) for i in range(N)]
print("psi16 =", psi16, " (order", order(psi16, q), ")")
print("GMb   =", GMb)


psi16 = 3  (order 16 )
GMb   = [1, 13, 9, 15, 3, 5, 10, 11]


**Sanity check** — confirm this is a correct negacyclic NTT via the convolution property $\mathrm{NTT}(a*b \bmod x^N{+}1)[k] = \mathrm{NTT}(a)[k]\cdot\mathrm{NTT}(b)[k]$.

In [180]:
def mq_add(x, y):
    d = x + y - q
    return d + q if d < 0 else d

def mq_sub(x, y):
    d = x - y
    return d + q if d < 0 else d

def mq_NTT_values(a):
    '''Plain in-place negacyclic forward NTT (no leakage).'''
    a = list(a); n = len(a); t = n; m = 1; gi = 1
    while m < n:
        ht = t >> 1
        for grp in range(m):
            j1 = grp * t; s = GMb[gi]; gi += 1
            for j in range(j1, j1 + ht):
                u = a[j]; v = (a[j + ht] * s) % q
                a[j] = mq_add(u, v); a[j + ht] = mq_sub(u, v)
        t = ht; m <<= 1
    return a

def negconv(f, g):
    n = len(f); r = [0] * n
    for i in range(n):
        for j in range(n):
            k = i + j; c = (f[i] * g[j]) % q
            if k < n: r[k] = (r[k] + c) % q
            else:     r[k - n] = (r[k - n] - c) % q
    return [x % q for x in r]

rng = np.random.default_rng(0); ok = True
for _ in range(500):
    f = list(rng.integers(0, q, N)); g = list(rng.integers(0, q, N))
    F, G, H = mq_NTT_values(f), mq_NTT_values(g), mq_NTT_values(negconv(f, g))
    if any((F[k] * G[k]) % q != H[k] for k in range(N)):
        ok = False; break
print("negacyclic NTT convolution property holds:", ok)


negacyclic NTT convolution property holds: True


## 2 · Step 1 — In-place modular reduction flags (the leaky `mq_NTT`)

FALCON's butterfly computes $v = a_2 w \bmod q$, $a_1' = (a_1+v)\bmod q$, $a_2' = (a_1-v)\bmod q$, each
via a constant-time conditional subtraction. The side channel (Fig. 2 of the paper) reveals *whether that
subtraction fired* — three bits per butterfly:

* `tw_norm`  — did $a_2 w$ exceed $q$?   (`[a2*w >= q]`)
* `add_norm` — did the addition reduce?   (`[a1 + v >= q]`)
* `sub_norm` — did the subtraction borrow? (`[a1 < v]`)

`generate_flags` runs the transform on the *victim* key and records these flags — this stands in for one
real power trace. We also print how often each flag fires: at $q=17$ they are well balanced (none stuck
near 0 % or 100 %), which is exactly what makes the attack informative.

In [181]:
def generate_flags(poly):
    '''Leaky forward NTT: returns {(stage,bfly): (tw_norm, add_norm, sub_norm)} and the NTT output.'''
    a = list(poly); n = len(a); t = n; m = 1; gi = 1; stage = 0
    flags = {}
    while m < n:
        ht = t >> 1; bfly = 0
        for grp in range(m):
            j1 = grp * t; s = GMb[gi]; gi += 1
            for j in range(j1, j1 + ht):
                u = a[j]; x = a[j + ht]
                raw_tw = x * s;  v   = raw_tw % q;        tw_norm  = int(raw_tw  != v)
                raw_add = u + v; a1p = mq_add(u, v);      add_norm = int(raw_add != a1p)
                raw_sub = u - v; a2p = mq_sub(u, v);      sub_norm = int(raw_sub != a2p)
                a[j] = a1p; a[j + ht] = a2p
                flags[(stage, bfly)] = (tw_norm, add_norm, sub_norm); bfly += 1
        t = ht; stage += 1; m <<= 1
    return flags, a

f_demo, _ = generate_flags(demo)
print("flags for the sample key (stage, bfly) -> (tw, add, sub):")
for k in sorted(f_demo): print("  ", k, "->", f_demo[k])


flags for the sample key (stage, bfly) -> (tw, add, sub):
   (0, 0) -> (0, 0, 0)
   (0, 1) -> (0, 0, 0)
   (0, 2) -> (1, 0, 1)
   (0, 3) -> (0, 1, 0)
   (1, 0) -> (1, 0, 1)
   (1, 1) -> (1, 0, 1)
   (1, 2) -> (1, 0, 1)
   (1, 3) -> (1, 0, 1)
   (2, 0) -> (1, 0, 0)
   (2, 1) -> (1, 0, 0)
   (2, 2) -> (1, 1, 0)
   (2, 3) -> (1, 1, 1)


## 3 · Butterfly wiring

At stage $s$ there are $2^s$ groups of size $N/2^s$; butterfly $b$ reads input wires $(i_a, i_b)$ with
$i_b = i_a + \tfrac{N}{2^{s+1}}$ and writes its outputs back to the same two wires (in-place).
`consumer` is the inverse map used by the backward pass: which later-stage butterfly consumes a given
wire, and as its `a1` (column 0) or `a2` (column 1) input.

In [182]:
def fwd_indices(stage, bfly):
    gs  = N >> stage; bpg = gs >> 1
    grp = bfly // bpg; off = bfly % bpg
    ia  = grp * gs + off; ib = ia + (gs >> 1)
    return ia, ib, grp

def consumer(stage1, wire):
    '''At `stage1`, which butterfly consumes `wire`, and column (0=a1, 1=a2).'''
    gs = N >> stage1; half = gs >> 1
    grp = wire // gs;  pos = wire % gs; bpg = gs >> 1
    return (grp * bpg + pos, 0) if pos < half else (grp * bpg + (pos - half), 1)


## 4 · Step 2 — Forward pruning (Algorithm 2)

Every secret coefficient starts with the attacker's prior: all candidate values in
$\{-B,\dots,B\}\bmod q$. For each butterfly we enumerate all input pairs $(a_1,a_2)$, keep only those
whose three computed flags match the measured flags, and propagate the surviving outputs $a_1',a_2'$
forward as the next stage's candidate sets. Each butterfly's surviving rows `(a1, a2, a1', a2')` are
stored for the backward pass.

> At $q = 17$ every candidate set is bounded by $q$, so the cross products stay tiny and the whole thing
> runs instantly. This is also where the real attack's cost comes from: at $q = 12289$ the output values
> spread across $[0,q)$ and the sets explode — which is why the full-scale attack needs a GPU and ~1.5 TB
> of disk.

In [183]:
def butterfly(a1_range, a2_range, w, f_tw, f_add, f_sub):
    '''All surviving rows (a1, a2, a1', a2') for one butterfly, given measured flags.'''
    a1 = np.asarray(a1_range, dtype=np.int64).reshape(-1, 1)      # (A,1)
    a2 = np.asarray(a2_range, dtype=np.int64).reshape(1, -1)      # (1,B)
    if a1.size == 0 or a2.size == 0:
        return np.empty((0, 4), dtype=np.int64)
    A, Bn = a1.shape[0], a2.shape[1]
    raw_tw  = a2 * w; red_tw = raw_tw % q                          # twiddle multiply (1,B)
    raw_add = a1 + red_tw; red_add = raw_add % q                   # (A,B)
    raw_sub = a1 - red_tw; red_sub = raw_sub % q
    mask = (np.broadcast_to(raw_tw != red_tw, (A, Bn)) == bool(f_tw)) \
         & ((raw_add != red_add) == bool(f_add)) \
         & ((raw_sub != red_sub) == bool(f_sub))
    if not mask.any():
        return np.empty((0, 4), dtype=np.int64)
    a1b = np.broadcast_to(a1, (A, Bn))[mask]
    a2b = np.broadcast_to(a2, (A, Bn))[mask]
    a1p = np.broadcast_to(red_add, (A, Bn))[mask]
    a2p = np.broadcast_to(red_sub, (A, Bn))[mask]
    return np.stack([a1b, a2b, a1p, a2p], axis=1)


def forward_pass(flags, init_range):
    '''Algorithm 2: prune each butterfly by its flags, propagate marginal output sets forward.'''
    Aval = [[list(init_range) for _ in range(N)]] + [[None] * N for _ in range(logN)]
    traces = {}
    for stage in range(logN):
        for bfly in range(N // 2):
            ia, ib, grp = fwd_indices(stage, bfly)
            w = GMb[(1 << stage) + grp]
            ftw, fadd, fsub = flags[(stage, bfly)]
            tr = butterfly(Aval[stage][ia], Aval[stage][ib], w, ftw, fadd, fsub)
            traces[(stage, bfly)] = tr
            if stage < logN - 1:
                Aval[stage + 1][ia] = np.unique(tr[:, 2]).tolist() if tr.size else []
                Aval[stage + 1][ib] = np.unique(tr[:, 3]).tolist() if tr.size else []
    return traces


## 5 · Step 3 — Backward refinement (Algorithm 3)

Working from the later stages back toward the inputs, each butterfly's forward trace is filtered so its
outputs $a_1', a_2'$ keep only values that still survive as *inputs* of the butterflies that consume them
downstream. This propagates late-stage constraints back to the first stage. Afterwards, the distinct
surviving inputs of the stage-0 butterflies are the recovered candidate sets for the secret
coefficients.

In [184]:
def backward_pass(traces):
    def refined(stage1, bfly, col):
        tr = traces[(stage1, bfly)]
        return set(np.unique(tr[:, col]).tolist()) if tr.size else set()
    for stage in range(logN - 2, -1, -1):
        for bfly in range(N // 2):
            ia, ib, _ = fwd_indices(stage, bfly)
            b_ia, col_ia = consumer(stage + 1, ia)
            b_ib, col_ib = consumer(stage + 1, ib)
            R_ia = refined(stage + 1, b_ia, col_ia)
            R_ib = refined(stage + 1, b_ib, col_ib)
            tr = traces[(stage, bfly)]
            if tr.size:
                Ria = np.fromiter(R_ia, np.int64, len(R_ia))
                Rib = np.fromiter(R_ib, np.int64, len(R_ib))
                traces[(stage, bfly)] = tr[np.isin(tr[:, 2], Ria) & np.isin(tr[:, 3], Rib)]
    return traces

def recovered_sets(traces):
    rec = [None] * N
    for wi in range(N):
        b, col = consumer(0, wi)
        tr = traces[(0, b)]
        rec[wi] = sorted(set(np.unique(tr[:, col]).tolist())) if tr.size else []
    return rec

def run_attack(poly, B=B):
    '''Full NTT-PEEL guess-and-prune attack. Returns recovered sets + reduction metrics.'''
    init_range = [v % q for v in range(-B, B + 1)]
    flags, _ = generate_flags(poly)
    traces = backward_pass(forward_pass(flags, init_range))
    rec = recovered_sets(traces)
    counts = [len(r) for r in rec]
    sound = all((poly[i] % q) in set(rec[i]) for i in range(N))   # true key must always survive
    return dict(rec=rec, counts=counts, sound=sound,
                log2space=sum(math.log2(max(1, c)) for c in counts),
                fully=sum(1 for c in counts if c == 1))


## 6 · Step 4 — Attack a typical key

Run the full pipeline on one typical sampled key and look at the per-coefficient reduction.

In [185]:
typ = [v % q for v in [-2, 2, 1, 0, 0, 1, 0, 1]] 
r = run_attack(typ)
init_bits = N * math.log2(2 * B + 1)
print("secret (signed)        :", to_signed(typ))
print("per-coeff candidates   :", r['counts'], " (started at", 2 * B + 1, "each)")
print("coeffs uniquely found  :", r['fully'], "/", N)
print("search space 2^%.1f  ->  2^%.1f   (%.1f bits removed)   true key kept: %s"
      % (init_bits, r['log2space'], init_bits - r['log2space'], r['sound']))


secret (signed)        : [-2, 2, 1, 0, 0, 1, 0, 1]
per-coeff candidates   : [1, 1, 1, 1, 1, 1, 1, 1]  (started at 9 each)
coeffs uniquely found  : 8 / 8
search space 2^25.4  ->  2^0.0   (25.4 bits removed)   true key kept: True


## 7 · Step 5 — Explore the input-coefficient space

Different secret polynomials leak different amounts. We sample many σ-distributed keys (those with
$\max|c_i|\le B$, so the guess range is sound) and rank them by the residual search space after the
attack. The best key is the one the attack reduces the most — the "input coefficient set that gives the
best reduction". At $q = 17$ the best keys are pinned down **completely** (residual $2^0$, all 8
coefficients unique).

In [186]:
SEED_EXPLORE, M = 0, 512

random.seed(SEED_EXPLORE)
candidates = []
while len(candidates) < M:
    poly = [sample_coeff() for _ in range(N)]
    if max(abs(x) for x in to_signed(poly)) <= B:          # keep guess range sound
        candidates.append(poly)

# flag-balance diagnostic over the explored keys (confirms flags are NOT saturated)
fire = [0, 0, 0]; tot = 0
for poly in candidates:
    for v in generate_flags(poly)[0].values():
        fire[0] += v[0]; fire[1] += v[1]; fire[2] += v[2]; tot += 1
print(f"flag firing rate over {M} keys:  tw {100*fire[0]/tot:.0f}%   add {100*fire[1]/tot:.0f}%   sub {100*fire[2]/tot:.0f}%")

t0 = time.time()
scored = []
for poly in candidates:
    res = run_attack(poly)
    assert res['sound']
    scored.append((res['log2space'], res['fully'], tuple(res['counts']), tuple(poly), res['rec']))
scored.sort(key=lambda x: (x[0], -x[1]))               # smallest residual, then most unique coeffs
dt = time.time() - t0

residuals = [s[0] for s in scored]
best = scored[0]
BEST_KEY = list(best[3]); BEST_REC = best[4]
full_recoveries = sum(1 for s in scored if s[1] == N)
print(f"\nexplored {M} keys in {dt:.1f}s   (initial space per key = 2^{init_bits:.2f})")
print(f"residual search space:  min 2^{min(residuals):.2f}   "
      f"   max 2^{max(residuals):.2f}")
print()
print("==> BEST key found:")
print("   signed coeffs          :", to_signed(BEST_KEY))
print("   mod-q  coeffs          :", BEST_KEY)
print("   per-coeff candidates   :", list(best[2]))
print("   uniquely recovered     :", best[1], "/", N)
print("   residual 2^%.2f  (from 2^%.2f  =>  %.1f bits removed)"
      % (best[0], init_bits, init_bits - best[0]))


flag firing rate over 512 keys:  tw 69%   add 37%   sub 46%

explored 512 keys in 0.2s   (initial space per key = 2^25.36)
residual search space:  min 2^0.00      max 2^17.29

==> BEST key found:
   signed coeffs          : [0, 1, 0, -2, 1, 0, -1, -3]
   mod-q  coeffs          : [0, 1, 0, 15, 1, 0, 16, 14]
   per-coeff candidates   : [1, 1, 1, 1, 1, 1, 1, 1]
   uniquely recovered     : 8 / 8
   residual 2^0.00  (from 2^25.36  =>  25.4 bits removed)


Distribution of how much the attack reduces the search space across the explored keys — the chosen best key sits at the far (full-recovery) end.

## 8 · Step 6 — Finishing the job with a joint consistency check

Guess-and-prune propagates *marginal* (per-coefficient) sets, so it can leave a few candidates even when
the key is uniquely determined jointly. A final **joint** check — enumerate the cross product of the
surviving per-coefficient sets and keep only polynomials whose flags match the trace — collapses the
remaining ambiguity. The best key needs no finishing (it is already unique after pruning), so we
demonstrate the step on the *typical* key from Step 5.

In [187]:
def joint_recover(poly):
    rec = run_attack(poly)['rec']
    tgt, _ = generate_flags(poly)
    combos = math.prod(len(s) for s in rec)
    consistent = [c for c in itertools.product(*rec) if generate_flags(list(c))[0] == tgt]
    exact = [len({c[i] for c in consistent}) for i in range(N)]
    return rec, combos, consistent, exact

rec, combos, consistent, exact = joint_recover(typ)
print("typical key (signed)            :", to_signed(typ))
print("guess-and-prune per-coeff counts:", [len(s) for s in rec])
print(f"joint space to check            : {combos} combos (2^{math.log2(combos):.2f})")
print(f"exact flag-consistent keys      : {len(consistent)} (2^{math.log2(len(consistent)):.2f})")
print("true key among them             :", tuple(typ) in set(consistent))
print("coefficients pinned EXACTLY      :", sum(1 for e in exact if e == 1), "/", N)

print("\nbest key (signed)               :", to_signed(BEST_KEY))
print("best-key per-coeff counts        :", [len(s) for s in BEST_REC],
      "-> already unique, no joint step needed" if all(len(s) == 1 for s in BEST_REC) else "")
print("\nIn the full FALCON-512 attack any residual is resolved with lattice reduction against the")
print("public key h = g*f^{-1} mod q (out of scope for this toy).")


typical key (signed)            : [-2, 2, 1, 0, 0, 1, 0, 1]
guess-and-prune per-coeff counts: [1, 1, 1, 1, 1, 1, 1, 1]
joint space to check            : 1 combos (2^0.00)
exact flag-consistent keys      : 1 (2^0.00)
true key among them             : True
coefficients pinned EXACTLY      : 8 / 8

best key (signed)               : [0, 1, 0, -2, 1, 0, -1, -3]
best-key per-coeff counts        : [1, 1, 1, 1, 1, 1, 1, 1] -> already unique, no joint step needed

In the full FALCON-512 attack any residual is resolved with lattice reduction against the
public key h = g*f^{-1} mod q (out of scope for this toy).


## 9 · Correctness check — guess-and-prune vs. exhaustive brute force

For a small guess range the whole key space is enumerable, so we can prove the attack is correct: the
guess-and-prune candidate sets must be a **superset** of the exact flag-consistent marginals (it never
discards the truth) and must always contain the true coefficients.

In [188]:
def exhaustive_marginals(poly, Bsmall):
    tgt, _ = generate_flags(poly)
    vals = [v % q for v in range(-Bsmall, Bsmall + 1)]
    marg = [set() for _ in range(N)]; total = 0
    for cand in itertools.product(vals, repeat=N):
        if generate_flags(list(cand))[0] == tgt:
            total += 1
            for i, c in enumerate(cand): marg[i].add(c)
    return [sorted(s) for s in marg], total

random.seed(1)
Bsmall = 3                                   # 7^8 = 5.76M keys enumerated
kpoly =  [v % q for v in [2, 1, 1, 1, 1, 1, 1, 1] ]
bf_marg, bf_total = exhaustive_marginals(kpoly, Bsmall)
gp = run_attack(kpoly, Bsmall)
superset = all(set(bf_marg[i]).issubset(set(gp['rec'][i])) for i in range(N))
true_in  = all((kpoly[i] % q) in set(gp['rec'][i]) for i in range(N))
print("secret (signed)                 :", to_signed(kpoly))
print("exact flag-consistent keys      :", bf_total)
print("exhaustive per-coeff marginals  :", [len(s) for s in bf_marg])
print("guess-and-prune per-coeff counts:", gp['counts'])
print("GP is a superset of exact        :", superset)
print("true key inside GP output        :", true_in)


secret (signed)                 : [2, 1, 1, 1, 1, 1, 1, 1]
exact flag-consistent keys      : 1
exhaustive per-coeff marginals  : [1, 1, 1, 1, 1, 1, 1, 1]
guess-and-prune per-coeff counts: [3, 2, 2, 1, 1, 1, 1, 1]
GP is a superset of exact        : True
true key inside GP output        : True


## 10 · Notes, scope, and relation to the full artifact

* **Faithful parts:** the negacyclic CT/DIF butterfly network, the twiddle convention (bit-reversed
  powers of a primitive $2N$-th root, the same construction FALCON uses), the three in-place
  modular-reduction leakage flags, Gaussian key sampling, and the forward-pruning / backward-refinement
  guess-and-prune algorithm (Alg. 2 & 3 of the paper).
* **Downscaled parts:** $N = 8$ (3 stages, 4 butterflies/stage) and the modulus $q = 17$ instead of
  $12289$. The modulus must satisfy $q\equiv 1\pmod{16}$; among the valid small primes, $q=17$ keeps the
  reduction flags balanced and makes the attack over-determined, so the strongest keys are recovered
  completely. Everything runs on a CPU in seconds — no CUDA, no multi-TB storage (that cost only appears
  as $q$ and $N$ grow and the candidate sets spread across $[0,q)$).
* **Lattice reduction** (Step 4 of the original artifact) resolves any final residual using the public
  key; it is omitted here because it is only meaningful at full NTRU dimension.

**Citation**

> J. Qiu and A. Aysu, *NTT-PEEL: Bit Shift Side-Channel in FALCON's Number Theoretic Transform*,
> IACR TCHES 2026(2), 709-735. DOI:10.46586/tches.v2026.i2.709-735

